# Algorithm 2 — Turn-level Friction & Repair Analysis (TFRA)

**File:** `src/CopilotScope.Collector/Quality/SegmentAnalyzer.cs`

Each agent-invocation trace is a **turn**.  Every turn starts at a perfect score of 1.0 and
accumulates penalty deductions based on observable telemetry.

---

## 1  Per-turn Score

$$
\tau_i = \max\!\left(0,\; 1
  - 0.35 \cdot \min(e^{\text{LLM}}_i, 2)
  - 0.15 \cdot \min(e^{\text{tool}}_i, 3)
  - \Delta^{\text{latency}}_i
  - \Delta^{\text{repair}}_i
\right)
$$

### 1.1  Latency penalty $\Delta^{\text{latency}}_i$

Latency is **self-referential**: a turn is penalised only when it is slow *relative to its own
session*, not relative to an absolute threshold.  Let $m$ = session median TTFT.

$$
\Delta^{\text{latency}}_i =
\begin{cases}
  0.30 & \text{if } \text{TTFT}_i \geq 3m \\
  0.15 & \text{if } 1.5m \leq \text{TTFT}_i < 3m \\
  0    & \text{otherwise}
\end{cases}
$$

### 1.2  Repair-loop penalty $\Delta^{\text{repair}}_i$

A repair loop is detected when three conditions hold simultaneously:

$$
\Delta^{\text{repair}}_i = 0.10 \cdot \mathbf{1}\!\left[
  e^{\text{tool}}_i > 0
  \;\wedge\;
  \frac{c^{\text{tool}}_i}{\max(1, c^{\text{LLM}}_i)} \geq 2 \cdot m^{\text{tool-ratio}}
  \;\wedge\;
  c^{\text{tool}}_i \geq 3
\right]
$$

where $m^{\text{tool-ratio}}$ = session median of $(c^{\text{tool}}_j / \max(1, c^{\text{LLM}}_j))$.

---

## 2  Session Median (used for both latency and tool-ratio)

$$
\text{Median}(V) =
\begin{cases}
  v_{\lfloor |V|/2 \rfloor} & \text{if } |V| \text{ is odd} \\
  \dfrac{v_{|V|/2 - 1} + v_{|V|/2}}{2} & \text{if } |V| \text{ is even}
\end{cases}
\quad (V \text{ sorted ascending})
$$

---

## 3  Worst/Best Selection

- **Worst turn:** $\arg\min_i \tau_i$, ties broken by $\arg\max_i \text{TTFT}_i$
- **Best turn:** $\arg\max_i \tau_i$, ties broken by $\arg\min_i \text{TTFT}_i$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from statistics import median

@dataclass
class Turn:
    index: int
    chat_calls: int
    chat_errors: int
    tool_calls: int
    tool_errors: int
    ttft_ms: float  # average TTFT for this turn

def analyze_turns(turns: list[Turn]):
    active = [t for t in turns if t.chat_calls + t.tool_calls > 0]
    if not active:
        return []

    ttft_values = [t.ttft_ms for t in active if t.ttft_ms > 0]
    median_ttft = median(ttft_values) if ttft_values else 0

    tool_ratios = [t.tool_calls / max(1.0, t.chat_calls) for t in active]
    median_tool_ratio = median(tool_ratios)

    results = []
    for t in active:
        score = 1.0
        reasons = []

        # LLM error penalty
        if t.chat_errors > 0:
            score -= 0.35 * min(t.chat_errors, 2)
            reasons.append(f'{t.chat_errors} LLM error(s)')

        # Tool error penalty
        if t.tool_errors > 0:
            score -= 0.15 * min(t.tool_errors, 3)
            reasons.append(f'{t.tool_errors} tool error(s)')

        # Latency penalty (self-referential)
        if median_ttft > 0 and t.ttft_ms > 0:
            ratio = t.ttft_ms / median_ttft
            if ratio >= 3.0:
                score -= 0.30
                reasons.append(f'TTFT {ratio:.1f}x session median')
            elif ratio >= 1.5:
                score -= 0.15
                reasons.append(f'TTFT {ratio:.1f}x session median')

        # Repair-loop penalty
        tool_ratio = t.tool_calls / max(1.0, t.chat_calls)
        if (t.tool_errors > 0 and median_tool_ratio > 0
                and tool_ratio >= 2 * median_tool_ratio
                and t.tool_calls >= 3):
            score -= 0.10
            reasons.append(f'repair loop ({t.tool_calls} tools for {t.chat_calls} chat)')

        if not reasons:
            reasons.append('clean turn')

        results.append({'turn': t.index + 1, 'score': max(0.0, score),
                        'ttft': t.ttft_ms, 'reasons': ', '.join(reasons)})
    return results

# --- Unit results ---
sessions = {
    'Scenario A — clean session': [
        Turn(0, 2, 0, 3, 0, 400),
        Turn(1, 3, 0, 4, 0, 380),
        Turn(2, 2, 0, 2, 0, 420),
    ],
    'Scenario B — LLM error + latency spike': [
        Turn(0, 2, 0, 3, 0, 400),
        Turn(1, 2, 2, 3, 0, 1500),  # 2 LLM errors
        Turn(2, 2, 0, 3, 0, 380),
        Turn(3, 2, 0, 3, 0, 390),
    ],
    'Scenario C — repair loop': [
        Turn(0, 2, 0, 2, 0, 300),
        Turn(1, 1, 0, 6, 2, 350),   # many tool calls + errors → repair loop
        Turn(2, 2, 0, 2, 0, 310),
    ],
}

for name, turns in sessions.items():
    print(f'\n{name}')
    print(f'{"Turn":>5} {"Score":>7} {"TTFT ms":>9}  Reasons')
    for r in analyze_turns(turns):
        print(f'{r["turn"]:>5} {r["score"]:>7.3f} {r["ttft"]:>9.0f}  {r["reasons"]}')

In [ ]:
# Penalty decomposition bar chart for Scenario B
turns_b = [
    Turn(0, 2, 0, 3, 0, 400),
    Turn(1, 2, 2, 3, 0, 1500),
    Turn(2, 2, 0, 3, 0, 380),
    Turn(3, 2, 0, 3, 0, 390),
]
results_b = analyze_turns(turns_b)
labels  = [f'Turn {r["turn"]}' for r in results_b]
scores  = [r['score'] for r in results_b]
deficits = [1 - s for s in scores]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x, scores,   label='Score', color='steelblue')
ax.bar(x, deficits, bottom=scores, label='Penalty', color='salmon', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score (0–1)')
ax.set_title('TFRA — penalty decomposition (Scenario B)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('tfra_penalty_chart.png', dpi=150)
plt.show()